<a href="https://colab.research.google.com/github/sohan-paul/Qiskit-fall-fest-portfolio-optimization/blob/main/sp_advanced_challenge2_qiskit_portfolio_optimizatin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install qiskit qiskit-optimization


In [ ]:
import numpy as np


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
from qiskit_optimization.problems import QuadraticProgram

In [ ]:
n=20

In [ ]:
np.random.seed(0)
mu=np.random.uniform(0.005,0.015,n)

In [ ]:
sigma=np.random.uniform(-0.05,0.5,(n,n))

In [ ]:
sigma=np.dot(sigma,sigma.T)

In [ ]:
q=1.5

In [ ]:
k=10

In [ ]:
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit_optimization.problems import QuadraticProgram

qp = QuadraticProgram()

for i in range(n):
    qp.binary_var(name=f'x_{i}')

qp.minimize(linear=-(1 - q) * mu, quadratic=q * sigma)

# Add the budget constraint
qp.linear_constraint(linear=[1] * n, sense="==", rhs=k)
print("Original Model:\n", qp.export_as_lp_string())

# Convert the constrained problem into an unconstrained QUBO
qubo_converter = QuadraticProgramToQubo()
qubo_model = qubo_converter.convert(qp)

# Convert the unconstrained QUBO to an Ising Hamiltonian
qubit_op, offset = qubo_model.to_ising()

print("Successfully converted to Ising Hamiltonian!")

Original Model:
 \ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: CPLEX

Minimize
 obj: 0.005244067520 x_0 + 0.006075946832 x_1 + 0.005513816880 x_2
      + 0.005224415915 x_3 + 0.004618273997 x_4 + 0.005729470565 x_5
      + 0.004687936056 x_6 + 0.006958865004 x_7 + 0.007318313803 x_8
      + 0.004417207594 x_9 + 0.006458625190 x_10 + 0.005144474599 x_11
      + 0.005340222805 x_12 + 0.007127983191 x_13 + 0.002855180291 x_14
      + 0.002935646499 x_15 + 0.002601091987 x_16 + 0.006663099228 x_17
      + 0.006390783755 x_18 + 0.006850060741 x_19 + [ 5.438551050412 x_0^2
      + 4.931141465410 x_0*x_1 + 3.725611971622 x_0*x_2 + 5.501140463362 x_0*x_3
      + 9.508723617304 x_0*x_4 + 7.673590032071 x_0*x_5 + 9.645009396041 x_0*x_6
      + 6.662538721985 x_0*x_7 + 5.267478162801 x_0*x_8 + 6.527016301689 x_0*x_9
      + 7.217433912670 x_0*x_10 + 6.929275991112 x_0*x_11
      + 9.101113391959 x_0*x_12 + 6.781259255913 x_0*x_13
      + 5.872471148360 x_0*x_14 + 

/tmp/ipykernel_3861/900751561.py:13: DeprecationWarning: The method ``qiskit_optimization.problems.quadratic_program.QuadraticProgram.export_as_lp_string()`` is deprecated as of Qiskit 0.7.0. It will be removed no earlier than 3 months after the release date. Use prettyprint instead.
  print("Original Model:\n", qp.export_as_lp_string())


In [ ]:
!pip install qiskit-optimization

In [ ]:
!pip install qiskit-algorithms

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 7.2 MB/s eta 0:00:00


In [ ]:
from qiskit_algorithms import NumPyMinimumEigensolver, QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization.algorithms import MinimumEigenOptimizer, WarmStartQAOAOptimizer
from qiskit.primitives import StatevectorSampler
from qiskit.circuit import QuantumCircuit

# A simple, robust V2-to-V1 Sampler adapter wrapper to keep compatibility with qiskit-algorithms
class V1SamplerAdapter:
    def __init__(self):
        self._sampler = StatevectorSampler()

    def run(self, circuits, parameter_values=None, **run_options):
        # Standardize inputs into modern V2 Primitive Unified Blocks (PUBs)
        pubs = []

        # Handle different structures sent by different algorithms solvers
        if isinstance(circuits, list):
            for item in circuits:
                if isinstance(item, tuple):
                    # If already packaged as (circuit, parameters)
                    pubs.append(item)
                else:
                    pubs.append((item, parameter_values if parameter_values is not None else []))
        elif isinstance(circuits, tuple):
            # If it's a single tuple of (circuit, parameters)
            if len(circuits) == 2 and isinstance(circuits[0], QuantumCircuit):
                pubs.append(circuits)
            else:
                pubs.append((circuits, parameter_values if parameter_values is not None else []))
        else:
            pubs.append((circuits, parameter_values if parameter_values is not None else []))

        job = self._sampler.run(pubs)

        # Standardize results to resemble V1 expectations
        class V1JobMock:
            def __init__(self, v2_job):
                self.v2_job = v2_job
            def result(self):
                v2_result = self.v2_job.result()

                # Helper class to mock the data.meas hierarchical interface
                class MeasMock:
                    def __init__(self, pub_result):
                        self._pub_result = pub_result
                        self.num_shots = pub_result.data.meas.num_shots
                    def get_int_counts(self):
                        return self._pub_result.data.meas.get_int_counts()
                    def get_counts(self):
                        return self._pub_result.data.meas.get_counts()

                class DataMock:
                    def __init__(self, pub_result):
                        self.meas = MeasMock(pub_result)

                class PubResultMock:
                    def __init__(self, pub_result):
                        self.data = DataMock(pub_result)

                class V1ResultMock(list):
                    def __init__(self, r):
                        mocks = [PubResultMock(pub_res) for pub_res in r]
                        super().__init__(mocks)
                        self.quasi_dists = [pub_res.data.meas.get_counts() for pub_res in r]

                return V1ResultMock(v2_result)

        return V1JobMock(job)

# Set up the exact classical solver (pre-solver)
exact_solver = NumPyMinimumEigensolver()
pre_optimizer = MinimumEigenOptimizer(exact_solver)

# Set up the standard QAOA using our V1-adapted Sampler
qaoa = QAOA(sampler=V1SamplerAdapter(), optimizer=COBYLA(), reps=1)

# Set up the Warm Start QAOA Optimizer
optimizer = WarmStartQAOAOptimizer(
    pre_solver=pre_optimizer,
    relax_for_pre_solver=False,
    qaoa=qaoa
)

# Solve the unconstrained QUBO model using the warm-start strategy
result = optimizer.solve(qubo_model)

# Display the optimal portfolio selection
print("Optimal portfolio selection:", result.x)
print(f"Total number of stocks selected: {sum(result.x)} (Target: {k})")

/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate expected returns and risks for all assets
returns = mu
risks = np.sqrt(np.diag(sigma))

# Create scatter plot of risk vs return
plt.figure(figsize=(10, 6))
plt.scatter(risks, returns, color="blue", label="All Assets")

# Highlight selected stocks
selected_indices = np.where(result.x == 1)[0]
plt.scatter(
    risks[selected_indices],
    returns[selected_indices],
    color="red",
    s=100,
    label="Selected Portfolio",
)

# Add labels and title
plt.title("Portfolio Optimization: Risk vs. Expected Return")
plt.xlabel("Risk (Standard Deviation)")
plt.ylabel("Expected Return")
plt.axhline(0, color="grey", linewidth=0.5)
plt.axvline(0, color="grey", linewidth=0.5)
plt.grid(True, linestyle="--")
plt.legend()
plt.show()

In [ ]:
!pip install qiskit-aer

In [ ]:
import numpy as np

# Simple Greedy Algorithm: Select the K stocks with the highest expected return (mu)
# np.argsort sorts in ascending order, so we take the last 'k' elements for the highest values
top_k_indices = np.argsort(mu)[-k:][::-1]

# Create a binary array to match the format of your QAOA output
classical_greedy_selection = np.zeros(n)
classical_greedy_selection[top_k_indices] = 1

print("--- Classical Greedy Method ---")
print("Optimal portfolio selection:", classical_greedy_selection)
print(f"Total number of stocks selected: {sum(classical_greedy_selection)} (Target: {k})")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit_optimization.converters import LinearEqualityToPenalty

# NOTE: If your original constrained quadratic program variable is named
# something other than 'qp' (e.g., 'problem'), change it on line 12.

lambda_values = np.linspace(0.5, 10, 20) # Sweeping lambda from 0.5 to 10
fraction_valid = []

for lam in lambda_values:
    # 1. Convert the constrained problem to a QUBO using the current lambda penalty
    converter = LinearEqualityToPenalty(penalty=lam)
    qubo_lam = converter.convert(qp)

    # 2. Solve using your already defined WarmStartQAOAOptimizer
    result_lam = optimizer.solve(qubo_lam)

    # 3. Calculate the fraction (probability) of samples that buy exactly 'k' stocks
    valid_probability = 0
    for sample in result_lam.samples:
        if sum(sample.x) == k:
            valid_probability += sample.probability

    fraction_valid.append(valid_probability)

# 4. Plot the results
plt.figure(figsize=(10, 6))
plt.plot(lambda_values, fraction_valid, marker='o', color='purple')
plt.xlabel('Penalty Parameter ($\lambda$)')
plt.ylabel('Fraction of Valid Samples (Exactly K stocks)')
plt.title('Parameter Sweep: Fraction of Valid Portfolios vs $\lambda$')
plt.grid(True)
plt.show()

In [ ]:
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import Sampler
from qiskit_optimization.algorithms import MinimumEigenOptimizer

# Initialize the optimizer and the reference Sampler primitive
optimizer = COBYLA(maxiter=250)
sampler = Sampler()

# Configure the QAOA algorithm (using reps=2 for a balance of speed and accuracy)
qaoa = QAOA(sampler=sampler, optimizer=optimizer, reps=2)

# Wrap QAOA in the MinimumEigenOptimizer
qaoa_optimizer = MinimumEigenOptimizer(qaoa)

# Solve the QuadraticProgram (qp) defined in your previous cell
result = qaoa_optimizer.solve(qp)

print("--- QAOA Optimization Results ---")
print(f"Optimal Selection (Binary): {result.x}")
print(f"Optimal Objective Value: {result.fval}")
print("\nDetailed output:\n", result.prettyprint())

In [ ]:
import numpy as np
from qiskit_optimization import QuadraticProgram
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import Sampler
from qiskit_optimization.algorithms import MinimumEigenOptimizer

# 1. Initialize 20 Stocks and Sectors for the Advanced Challenge
n_adv = 20
np.random.seed(42) # Ensures reproducibility

# Simulate expected returns and a symmetric covariance (risk) matrix
mu_adv = np.random.uniform(0.05, 0.2, n_adv)
sigma_adv = np.random.uniform(0.01, 0.1, (n_adv, n_adv))
sigma_adv = np.dot(sigma_adv, sigma_adv.transpose())

# Group the 20 stocks into 4 sectors
sectors = {
    'Tech': [0, 1, 2, 3, 4],
    'Finance': [5, 6, 7, 8, 9],
    'Energy': [10, 11, 12, 13, 14],
    'Healthcare': [15, 16, 17, 18, 19]
}

# 2. Build the Advanced Quadratic Program
qp_adv = QuadraticProgram()

# Add binary variables for all 20 stocks
for i in range(n_adv):
    qp_adv.binary_var(f"x_{i}")

# Objective: Maximize Return - Risk
risk_penalty = 0.5
linear_terms = {f"x_{i}": mu_adv[i] for i in range(n_adv)}
quadratic_terms = {}
for i in range(n_adv):
    for j in range(n_adv):
        quadratic_terms[(f"x_{i}", f"x_{j}")] = -risk_penalty * sigma_adv[i, j]

qp_adv.maximize(linear=linear_terms, quadratic=quadratic_terms)

# Constraint 1: Budget Rule (Select exactly 8 stocks out of 20)
budget = 8
qp_adv.linear_constraint(
    linear={f"x_{i}": 1 for i in range(n_adv)},
    sense="==",
    rhs=budget,
    name="total_budget"
)

# Constraint 2: Harder Sector Rule (At most 2 stocks per sector)
max_per_sector = 2
for sector_name, stock_indices in sectors.items():
    sector_linear = {f"x_{i}": 1 for i in stock_indices}
    qp_adv.linear_constraint(
        linear=sector_linear,
        sense="<=",
        rhs=max_per_sector,
        name=f"max_2_{sector_name}"
    )

# 3. Configure and Run QAOA
# Using reps=1 and maxiter=50 to ensure it finishes executing on a local simulator
optimizer_adv = COBYLA(maxiter=50)
sampler_adv = Sampler()
qaoa_adv = QAOA(sampler=sampler_adv, optimizer=optimizer_adv, reps=1)

qaoa_optimizer_adv = MinimumEigenOptimizer(qaoa_adv)

print("Executing Advanced QAOA for 20 stocks... (This may take a few minutes)")
result_adv = qaoa_optimizer_adv.solve(qp_adv)

# 4. Output the Final Advanced Portfolio
print("\n--- Advanced Utility Scale Results ---")
print(f"Optimal Binary Selection: {result_adv.x}")
print(f"Optimal Objective Value: {result_adv.fval}\n")

print("Selected Portfolio:")
for i in range(n_adv):
    if result_adv.x[i] == 1:
        # Find which sector this stock belongs to
        stock_sector = [s for s, idxs in sectors.items() if i in idxs][0]
        print(f"- Stock_{i} ({stock_sector})")